In [ ]:
%pip install kaggle
%pip install fastapi nest-asyncio uvicorn python-multipart pillow transformers torch pyyaml requests omegaconf

In [ ]:
yaml_config = """
              model_path: "google/vit-base-patch16-224"
              top_k: 5
              """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# Build and Initialize Model

In [ ]:
import io
import torch
import torch.nn.functional as F
from PIL import Image
from omegaconf import OmegaConf
from transformers import ViTImageProcessor, ViTForImageClassification

class ImageClassifier:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = ViTImageProcessor.from_pretrained(self.config.model_path)
        self.model = ViTForImageClassification.from_pretrained(self.config.model_path)
        self.model.eval()
        self.top_k = getattr(self.config, "top_k", 5)

    def predict(self, image_bytes):
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)

        top_probs, top_indices = torch.topk(probs, self.top_k, dim=-1)

        predictions = []
        for score, idx in zip(top_probs[0], top_indices[0]):
            predictions.append({
                "label": self.model.config.id2label[idx.item()],
                "confidence": round(float(score.item()), 4)
            })
        return {
            "top_prediction": predictions[0]["label"],
            "top_confidence": predictions[0]["confidence"],
            "top_k_predictions": predictions
        }


classifier = ImageClassifier("./config.yaml")

# Initialize API

In [ ]:
from pydantic import BaseModel
from fastapi import FastAPI, UploadFile, File, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from PIL import UnidentifiedImageError


class PredictionItem(BaseModel):
    label: str
    confidence: float

class PredictResponse(BaseModel):
    filename: str
    content_type: str
    top_prediction: str
    top_confidence: float
    top_k_predictions: list[PredictionItem]

app = FastAPI(title="ViT Image Classification API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get("/")
async def root():
    return {
        "service": "ViT Image Classification API",
        "model": "google/vit-base-patch16-224",
        "endpoints": ["GET /", "GET /health", "POST /predict"]
    }

@app.get("/health")
async def health():
    if classifier is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Model is not available"
        )
    return {
        "status": "healthy",
        "model_loaded": True,
    }

@app.post("/predict", response_model=PredictResponse)
async def predict(file: UploadFile = File(...)):
    if classifier is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Model is not ready"
        )

    types = ["image/jpeg", "image/png"]
    if file.content_type not in types:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Invalid file format: {file.content_type}. Only JPEG, PNG are supported"
        )

    image_bytes = await file.read()
    if len(image_bytes) == 0:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="The uploaded file is empty"
        )

    try:
        predictions = classifier.predict(image_bytes)
        return PredictResponse(
            filename=file.filename or "unknown", 
            content_type=file.content_type or "unknown",
            top_prediction=predictions["top_prediction"],
            top_confidence=predictions["top_confidence"],
            top_k_predictions=predictions["top_k_predictions"]
        )
    except UnidentifiedImageError:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="The file is corrupted or not a valid image format"
        )
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Error during model inference: {str(e)}"
        )



import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")